# 07 — Custom Baselines: RNA-only vs Protein-only

Models: `NaiveTissueDrugMeanPredictor`, `RandomForest`, `XGBoost` — architectures and
hyperparameter grids matched to drevalpy's own implementations (verified from the
installed 1.5.1 source):
- [`naive_pred.py`](https://github.com/daisybio/drevalpy/blob/development/drevalpy/models/baselines/naive_pred.py)
- [`multi_view_xgboost.py`](https://github.com/daisybio/drevalpy/blob/development/drevalpy/models/baselines/multi_view_xgboost.py)

Independent of drevalpy's experiment harness — loads `response_pairs.parquet` +
`data/splits/*.json` from notebook 04 directly. RNA-only and protein-only are run
and reported separately for every split/fold; fusion deferred. Multi-drug models
(global model across all pairs, drug identity via fingerprint) — not per-drug.

**Hyperparameter selection is now a one-time step, not per-fold:** grid search
(or a fixed dict you supply — toggle via `HPAM_MODE`) runs **once per (model, arm)**
on a single designated reference fold, and those hyperparameters are then reused
for every split/fold in the actual evaluation. This is a real shortcut from
drevalpy's own per-fold tuning, traded deliberately for speed — flag it if you'd
rather go back to per-fold search for the final numbers.

**One deviation from upstream, flagged explicitly:** `MultiViewXGBoost`'s yaml grid
defines `reg_lambda: [0.1]`, but `build_model()` never actually reads it — wired it
through properly here since the grid clearly intends to tune it.

## Environment setup (Colab or local)

In [ ]:
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !pip install -q xgboost rdkit
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = Path('/content/drive/MyDrive/multiomics_project')
else:
    BASE_PATH = Path('..')

print(f"Running on {'Colab' if IN_COLAB else 'local'} | BASE_PATH = {BASE_PATH}")

Running on local | BASE_PATH = ..


## Config

In [55]:
DATA_DIR = BASE_PATH / "data" / "GDSC2"
PROCESSED_DIR = BASE_PATH / "data" / "processed"
SPLITS_DIR = BASE_PATH / "data" / "splits"
RESULTS_DIR = BASE_PATH / "results" / "custom_baselines"

COL_CELL_LINE = "cell_line_name"
COL_DRUG = "drug_name"
COL_IC50 = "LN_IC50"
COL_CELLOSAURUS = "cellosaurus_id"
COL_TISSUE = "tissue"

SPLIT_TYPES = ["lco", "ldo", "lto"] #"lpo", 
ARMS = ["rna", "protein"]

VALIDATION_RATIO = 0.1
TOP_K_FEATURES = 1000   # matches drevalpy's proteomics_n_features default; applied to RNA too for symmetry
RANDOM_STATE = 42

# Quick smoke test before committing to the full run: limits to LCO, fold 0 only.
QUICK_TEST = False

# --- Hyperparameter mode ---
# "search": grid search once per (model, arm) on the reference fold below.
# "fixed":  skip search entirely, use FIXED_PARAMS as-is for every (model, arm).
HPAM_MODE = "fixed"  # "search" or "fixed"

GRID_SEARCH_SPLIT = "lco"
GRID_SEARCH_FOLD_INDEX = 0
SAVE_PREDICTIONS_FOLDS = {0}

# Only used when HPAM_MODE == "fixed". Must cover every model x arm combination below
# if you switch modes -- e.g. {"RandomForest": {"rna": {"max_depth": 10}, "protein": {"max_depth": 10}}, ...}
FIXED_PARAMS = {
    "RandomForest": {"rna": {"max_depth": 10}, "protein": {"max_depth": 10}},
    "XGBoost": {
        "rna": {"colsample_bytree": 0.8, "reg_alpha": 1, "reg_lambda": 0.1},
        "protein": {"colsample_bytree": 0.8, "reg_alpha": 1, "reg_lambda": 0.1},
    },
}

In [3]:
import warnings
warnings.filterwarnings("ignore")

import json
import time

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import ParameterGrid, train_test_split
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

## Load response pairs + splits (from notebook 04)

`pairs` and the split JSONs share the same 0..n-1 index space — that's the whole
point of how notebook 04 saved them.

In [59]:
pairs = pd.read_parquet(PROCESSED_DIR / "response_pairs.parquet")

splits = {}
for split_type in SPLIT_TYPES:
    with open(SPLITS_DIR / f"{split_type}.json") as f:
        splits[split_type] = json.load(f)

print(f"{len(pairs)} pairs loaded")
for split_type in SPLIT_TYPES:
    print(f"{split_type.upper()}: {len(splits[split_type])} folds")

176197 pairs loaded
LCO: 5 folds
LDO: 5 folds
LTO: 5 folds


## Load omics features and drug fingerprints

RNA/protein indexed by `cellosaurus_id` (same convention as notebook 04). Drug
fingerprints are Morgan fingerprints from SMILES (matches the `PLAN.md` approach) —
computed once and cached in a dict, since the same ~280 drugs repeat across all pairs.

In [34]:
rna = pd.read_csv(DATA_DIR / "gene_expression.csv", index_col=0)
protein = pd.read_csv(DATA_DIR / "proteomics.csv", index_col=0)
drug_smiles = pd.read_csv(DATA_DIR / "drug_smiles.csv")

rna = rna[~rna.index.duplicated(keep="first")].iloc[:,1:]
protein = protein[~protein.index.duplicated(keep="first")]

OMICS = {"rna": rna, "protein": protein}


def build_drug_fingerprints(drug_smiles_df: pd.DataFrame, radius: int = 2, n_bits: int = 2048) -> dict:
    """Morgan fingerprint per drug name. Drugs that fail to parse are skipped —
    any pair referencing them will fail loudly at feature-build time, not silently."""
    generator = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
    fps = {}
    for _, row in drug_smiles_df.iterrows():
        mol = Chem.MolFromSmiles(row["canonical_smiles"])
        if mol is None:
            continue
        fp = generator.GetFingerprint(mol)
        fps[row[COL_DRUG]] = np.array(fp, dtype=np.float32)
    return fps


drug_fp = build_drug_fingerprints(drug_smiles)
print(f"Fingerprints built for {len(drug_fp)} / {drug_smiles[COL_DRUG].nunique()} drugs")

Fingerprints built for 246 / 246 drugs


In [35]:
drug_fp

{'Romidepsin': array([0., 1., 0., ..., 0., 0., 0.], shape=(2048,), dtype=float32),
 'Paclitaxel': array([0., 1., 0., ..., 0., 0., 0.], shape=(2048,), dtype=float32),
 'Dactinomycin': array([0., 1., 0., ..., 0., 0., 0.], shape=(2048,), dtype=float32),
 'SN-38': array([0., 0., 0., ..., 0., 0., 0.], shape=(2048,), dtype=float32),
 'Docetaxel': array([0., 1., 0., ..., 0., 0., 0.], shape=(2048,), dtype=float32),
 'Bortezomib': array([0., 1., 0., ..., 0., 0., 0.], shape=(2048,), dtype=float32),
 'Rapamycin': array([0., 1., 0., ..., 0., 0., 0.], shape=(2048,), dtype=float32),
 'Camptothecin': array([0., 0., 0., ..., 0., 0., 0.], shape=(2048,), dtype=float32),
 'Vinorelbine': array([0., 0., 0., ..., 0., 0., 0.], shape=(2048,), dtype=float32),
 'GSK626616AC': array([0., 0., 0., ..., 0., 0., 0.], shape=(2048,), dtype=float32),
 'Vinblastine': array([0., 0., 0., ..., 0., 0., 0.], shape=(2048,), dtype=float32),
 'Daporinad': array([0., 0., 0., ..., 0., 0., 0.], shape=(2048,), dtype=float32),
 'Ele

In [36]:
protein

,P37108;SRP14_HUMAN,Q96JP5;ZFP91_HUMAN,Q9Y4H2;IRS2_HUMAN,P36578;RL4_HUMAN,Q6SPF0;SAMD1_HUMAN,O76031;CLPX_HUMAN,Q8WUQ7;CATIN_HUMAN,A6NIH7;U119B_HUMAN,Q9BTD8;RBM42_HUMAN,Q9P258;RCC2_HUMAN,...,P33151;CADH5_HUMAN,Q5EBL4;RIPL1_HUMAN,P49715;CEBPA_HUMAN,Q5TA45;INT11_HUMAN,O14924;RGS12_HUMAN,Q7Z3B1;NEGR1_HUMAN,O60669;MOT2_HUMAN,Q13571;LAPM5_HUMAN,Q96JM2;ZN462_HUMAN,P35558;PCKGC_HUMAN
cellosaurus_id,,,,,,,,,,,,,,,,,,,,,
CVCL_1762,6.828022,4.143455,2.237808,7.628785,3.198109,4.609018,NaN,2.470590,3.695348,5.707904,...,NaN,NaN,NaN,3.196077,NaN,NaN,NaN,NaN,NaN,NaN
CVCL_4384,7.014256,4.199872,2.440552,8.124585,NaN,4.768811,NaN,NaN,NaN,5.522833,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CVCL_X508,5.285911,3.357894,NaN,7.972680,NaN,4.520923,NaN,NaN,2.730884,4.294287,...,NaN,NaN,NaN,2.790234,NaN,NaN,NaN,NaN,NaN,NaN
CVCL_S976,5.707857,NaN,NaN,6.225738,NaN,4.495795,NaN,NaN,2.879809,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CVCL_C757,6.045911,3.693555,NaN,7.070925,3.495936,4.054377,NaN,NaN,3.442016,4.342368,...,NaN,3.022604,NaN,NaN,NaN,NaN,2.169524,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
CVCL_1081,6.664801,3.604150,NaN,6.994470,3.340518,3.804212,NaN,1.866548,3.279777,6.703322,...,NaN,NaN,NaN,2.863890,NaN,NaN,NaN,NaN,NaN,NaN
CVCL_1120,6.316308,4.869332,NaN,7.624330,3.926891,4.472120,NaN,NaN,3.484641,5.439826,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
CVCL_1240,6.230865,2.716858,NaN,7.235027,3.420646,4.717799,NaN,NaN,3.625137,6.155179,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
rna

,TSPAN6,TNMD,DPM1,SCYL3,C1orf112,FGR,CFH,FUCA2,GCLC,NFYA,...,LINC00526,PPY2,Unnamed: 17731,Unnamed: 17732,KRT18P55,Unnamed: 17734,POLRMTP1,UBL5P2,TBC1D3P5,Unnamed: 17738
cellosaurus_id,,,,,,,,,,,,,,,,,,,,,
CVCL_1104,7.632023,2.964585,10.379553,3.614794,3.380681,3.324692,3.566350,8.204530,5.235118,5.369039,...,6.786925,2.997054,3.109774,7.882377,3.331134,2.852537,3.130696,9.986616,3.073724,7.284733
CVCL_1174,7.548671,2.777716,11.807341,4.066887,3.732485,3.152404,7.827172,6.616972,5.809264,7.209653,...,5.317911,3.263745,3.059424,8.681302,2.992611,2.776771,3.260982,9.002814,3.000182,8.504804
CVCL_1110,8.712338,2.643508,9.880733,3.956230,3.236620,3.241246,2.931034,8.191246,5.426841,5.120747,...,3.143006,3.112145,2.930254,8.707886,2.886574,2.685307,3.176239,9.113243,2.916274,7.059092
CVCL_V001,7.797142,2.817923,9.883471,4.063701,3.558414,3.101247,7.211707,8.630643,5.617714,4.996434,...,3.153896,3.151576,2.850726,7.872535,3.812119,3.436412,3.074432,9.958284,3.256500,7.318125
CVCL_A555,7.729268,2.957739,10.418840,4.341500,3.840373,3.001802,3.375422,8.296950,5.669418,4.180205,...,3.652660,2.918475,2.849537,8.945953,3.412586,2.951270,3.213545,9.938978,3.396126,7.726867
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
CVCL_VS57,6.932178,2.622630,10.418475,3.947561,3.814831,3.170766,3.312925,9.408050,5.760336,4.531288,...,4.604765,3.217101,3.122624,9.031840,3.174122,2.846830,3.139208,10.018968,3.357660,7.135000
CVCL_2077,8.441628,2.639276,11.463742,4.425849,4.384732,3.229511,3.571204,8.193000,5.671600,4.943996,...,5.097882,3.102979,3.243552,8.110421,3.343723,2.959009,3.007502,9.332193,3.435411,10.392042
CVCL_2686,8.422922,2.879890,10.557777,3.550390,4.247189,3.176336,3.321811,8.901706,4.684851,4.215908,...,4.243448,3.034131,3.031143,9.161868,3.412558,2.974475,3.088841,10.742651,3.317945,6.203929


## Feature matrix builder

In [38]:
def select_top_variance_genes(arm: str, cell_line_ids: pd.Series, k: int) -> list[str]:
    """Top-k highest-variance columns, computed on the UNIQUE cell lines only (never
    duplicated per drug-pair) -- this is what avoids materializing a huge
    (n_pairs x n_genes) matrix before feature selection ever runs."""
    unique_ids = cell_line_ids.unique()
    sub = OMICS[arm].loc[unique_ids]
    return sub.var(axis=0).nlargest(k).index.tolist()


def build_feature_matrix(idx: np.ndarray, arm: str, selected_genes: list[str]) -> tuple[np.ndarray, np.ndarray]:
    sub = pairs.loc[idx]
    omics_X = OMICS[arm].loc[sub[COL_CELLOSAURUS], selected_genes].to_numpy(dtype=np.float32)
    drug_X = np.vstack([drug_fp[d] for d in sub[COL_DRUG]])
    X = np.hstack([omics_X, drug_X]).astype(np.float32)
    y = sub[COL_IC50].to_numpy()
    return X, y

## Group-aware validation split

Mirrors `_leave_group_out_cv` / `_leave_pair_out_cv` exactly: LPO gets a plain
random validation split; LCO/LDO/LTO split whole groups so validation never shares
a cell line/drug/tissue with the inner-train set. Used only during hyperparameter
selection now, not per-fold.

In [39]:
def make_validation_indices(train_idx: np.ndarray, split_type: str) -> tuple[np.ndarray, np.ndarray]:
    if split_type == "lpo":
        train_inner, val_idx = train_test_split(
            train_idx, test_size=VALIDATION_RATIO, shuffle=True, random_state=RANDOM_STATE
        )
        return np.array(train_inner), np.array(val_idx)

    group_col = {"lco": COL_CELLOSAURUS, "ldo": COL_DRUG, "lto": COL_TISSUE}[split_type]
    train_groups_all = pairs.loc[train_idx, group_col]
    unique_groups = train_groups_all.unique()
    train_groups, val_groups = train_test_split(
        unique_groups, test_size=VALIDATION_RATIO, shuffle=True, random_state=RANDOM_STATE
    )
    train_inner = train_idx[train_groups_all.isin(train_groups).to_numpy()]
    val_idx = train_idx[train_groups_all.isin(val_groups).to_numpy()]
    return train_inner, val_idx

## Hyperparameter grids and model factory

`RandomForest` grid matches drevalpy's yaml. `XGBoost` grid matches
`MultiViewXGBoost`'s yaml (`learning_rate=0.1`, `max_depth=10`, `subsample=0.8`
fixed; `colsample_bytree`, `reg_alpha`, `reg_lambda` tuned) — `n_estimators=100`
is drevalpy's hardcoded default (not in their yaml grid, so not tuned here either).

In [40]:
RF_GRID = {"max_depth": [5, 10, 30]}
XGB_GRID = {
    "colsample_bytree": [0.6, 0.8],
    "reg_alpha": [0, 1],
    "reg_lambda": [0.1],
}


def make_model(model_name: str, params: dict):
    if model_name == "RandomForest":
        return RandomForestRegressor(
            n_estimators=100, max_samples=0.2, criterion="squared_error",
            n_jobs=-1, random_state=RANDOM_STATE, **params,
        )
    if model_name == "XGBoost":
        return xgb.XGBRegressor(
            n_estimators=100, learning_rate=0.1, max_depth=10, subsample=0.8,
            random_state=RANDOM_STATE, n_jobs=-1, **params,
        )
    raise ValueError(model_name)


# def select_top_variance_features(X_ref: np.ndarray, k: int) -> np.ndarray:
#     """Indices of the top-k highest-variance omics columns, computed from X_ref only
#     (whatever counts as 'training data' at that stage) — never from val/test."""
#     n_omics = X_ref.shape[1] - 2048  # drug fingerprint is always the last 2048 cols
#     variances = X_ref[:, :n_omics].var(axis=0)
#     top_idx = np.argsort(variances)[-k:]
#     return np.concatenate([top_idx, np.arange(n_omics, X_ref.shape[1])])

## Hyperparameter selection — once per (model, arm)

Runs on a single reference fold (`GRID_SEARCH_SPLIT`/`GRID_SEARCH_FOLD_INDEX`),
**not** repeated for every fold of every split — that's the actual cost saving
versus the original per-fold design. If `HPAM_MODE == "fixed"`, this just returns
your `FIXED_PARAMS` dict and skips searching entirely.

In [41]:
def select_hyperparameters(model_name: str, arm: str) -> dict:
    if HPAM_MODE == "fixed":
        return FIXED_PARAMS[model_name][arm]

    fold = splits[GRID_SEARCH_SPLIT][GRID_SEARCH_FOLD_INDEX]
    train_idx = np.array(fold["train"])
    train_inner_idx, val_idx = make_validation_indices(train_idx, GRID_SEARCH_SPLIT)

    selected_genes = select_top_variance_genes(arm, pairs.loc[train_inner_idx, COL_CELLOSAURUS], TOP_K_FEATURES)
    X_train_inner, y_train_inner = build_feature_matrix(train_inner_idx, arm, selected_genes)
    X_val, y_val = build_feature_matrix(val_idx, arm, selected_genes)

    scaler = StandardScaler().fit(X_train_inner)
    X_train_inner_s = scaler.transform(X_train_inner)
    X_val_s = scaler.transform(X_val)

    grid = RF_GRID if model_name == "RandomForest" else XGB_GRID
    grid_list = list(ParameterGrid(grid))
    print(f"  [{model_name} | {arm}] searching {len(grid_list)} hyperparameter combinations "
          f"on {len(train_inner_idx)} train / {len(val_idx)} val rows...")

    best_rmse, best_params = np.inf, None
    for combo_i, params in enumerate(grid_list, 1):
        start = time.time()
        model = make_model(model_name, params)
        model.fit(X_train_inner_s, y_train_inner)
        rmse = root_mean_squared_error(y_val, model.predict(X_val_s))
        elapsed = time.time() - start
        marker = " <- best so far" if rmse < best_rmse else ""
        print(f"    [{combo_i}/{len(grid_list)}] {params} | val_rmse={rmse:.4f} | {elapsed:.1f}s{marker}")
        if rmse < best_rmse:
            best_rmse, best_params = rmse, params
    return best_params


## Fit + predict — fixed hyperparameters, full train → test

No inner validation split here anymore (that only happens once, above, during
selection) — this just fits on the entire training fold and predicts test.

In [42]:
def fit_predict(model_name: str, split_type: str, fold: dict, arm: str, params: dict) -> dict:
    train_idx, test_idx = np.array(fold["train"]), np.array(fold["test"])

    selected_genes = select_top_variance_genes(arm, pairs.loc[train_idx, COL_CELLOSAURUS], TOP_K_FEATURES)
    X_train, y_train = build_feature_matrix(train_idx, arm, selected_genes)
    X_test, y_test = build_feature_matrix(test_idx, arm, selected_genes)

    scaler = StandardScaler().fit(X_train)
    X_train_s = scaler.transform(X_train)
    X_test_s = scaler.transform(X_test)

    print(f"  fitting {model_name} on {len(train_idx)} train rows...", end=" ", flush=True)
    fit_start = time.time()
    model = make_model(model_name, params)
    model.fit(X_train_s, y_train)
    pred_test = model.predict(X_test_s)
    print(f"done ({time.time() - fit_start:.1f}s)")

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        r, _ = pearsonr(pred_test, y_test)
    rmse = root_mean_squared_error(y_test, pred_test)
    return {
        "n_train": len(train_idx), "n_test": len(test_idx),
        "pearson_r": r, "rmse": rmse,
        "test_idx": test_idx, "y_test": y_test, "pred_test": pred_test,
    }

## Naive baseline: `NaiveTissueDrugMeanPredictor`

Mean response per (tissue, drug) combination in train, falling back to the overall
train mean for combinations never seen in training — logic matched exactly to
[`naive_pred.py`](https://github.com/daisybio/drevalpy/blob/development/drevalpy/models/baselines/naive_pred.py).
No hyperparameters, no omics features — run once per split/fold (not duplicated
per arm, since it would be identical regardless of arm).

In [43]:
def fit_predict_naive_tissue_drug(split_type: str, fold: dict) -> dict:
    train_idx, test_idx = np.array(fold["train"]), np.array(fold["test"])
    train = pairs.loc[train_idx]
    test = pairs.loc[test_idx]

    dataset_mean = train[COL_IC50].mean()
    tissue_drug_means = train.groupby([COL_TISSUE, COL_DRUG])[COL_IC50].mean().to_dict()
    drug_means = train.groupby(COL_DRUG)[COL_IC50].mean().to_dict()
    tissue_means = train.groupby(COL_TISSUE)[COL_IC50].mean().to_dict()

    def predict_one(t, d):
        if (t, d) in tissue_drug_means:
            return tissue_drug_means[(t, d)]
        if d in drug_means:      # LDO: drug unseen as a (tissue,drug) pair, but tissue is known -> falls through to tissue_means below if drug also unseen
            return drug_means[d]
        if t in tissue_means:    # LTO: tissue unseen, but drug is known -> drug_means above already handles it; this is the LDO case (drug truly unseen too)
            return tissue_means[t]
        return dataset_mean

    pred_test = np.array([predict_one(t, d) for t, d in zip(test[COL_TISSUE], test[COL_DRUG])])
    y_test = test[COL_IC50].to_numpy()

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        r, _ = pearsonr(pred_test, y_test)
    rmse = root_mean_squared_error(y_test, pred_test)
    return {
        "n_train": len(train_idx), "n_test": len(test_idx),
        "pearson_r": r, "rmse": rmse,
        "test_idx": test_idx, "y_test": y_test, "pred_test": pred_test,
    }

## Run setup

Hyperparameters get selected once per (model, arm) here — printed so you can see
exactly what's being reused across every split/fold below. `QUICK_TEST=True`
limits the actual evaluation loops to LCO fold 0 only.

In [60]:
run_splits = {"lco": [splits["lco"][0]]} if QUICK_TEST else splits
results_rows = []
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Hyperparameter mode: {HPAM_MODE}")
selected_params = {}
for model_name in ["RandomForest", "XGBoost"]:
    selected_params[model_name] = {}
    for arm in ARMS:
        params = select_hyperparameters(model_name, arm)
        selected_params[model_name][arm] = params
        print(f"{model_name:13s} | {arm:8s} | {params}")


def save_predictions(split_type: str, arm: str, model_name: str, fold_i: int, out: dict) -> None:
    if fold_i not in SAVE_PREDICTIONS_FOLDS:
        return
    pred_dir = RESULTS_DIR / split_type / arm / model_name
    pred_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({
        "row_idx": out["test_idx"], "response": out["y_test"], "predictions": out["pred_test"],
    }).to_csv(pred_dir / f"predictions_fold_{fold_i}.csv", index=False)


def log_result(split_type: str, fold_i: int, arm: str, model_name: str, out: dict,
                elapsed: float, params: dict | None = None) -> None:
    results_rows.append({
        "split": split_type.upper(), "fold": fold_i, "arm": arm, "model": model_name,
        "n_train": out["n_train"], "n_test": out["n_test"],
        "pearson_r": round(out["pearson_r"], 4), "rmse": round(out["rmse"], 4),
        "params": params, "seconds": round(elapsed, 1),
    })
    print(f"{split_type.upper()} fold {fold_i} | {arm:8s} | {model_name:28s} "
          f"| r={out['pearson_r']:.3f} rmse={out['rmse']:.3f} | params={params} | {elapsed:.0f}s")

Hyperparameter mode: fixed
RandomForest  | rna      | {'max_depth': 10}
RandomForest  | protein  | {'max_depth': 10}
XGBoost       | rna      | {'colsample_bytree': 0.8, 'reg_alpha': 1, 'reg_lambda': 0.1}
XGBoost       | protein  | {'colsample_bytree': 0.8, 'reg_alpha': 1, 'reg_lambda': 0.1}


### Run — `NaiveTissueDrugMeanPredictor`

In [45]:
for split_type, folds in run_splits.items():
    for fold_i, fold in enumerate(folds):
        start = time.time()
        out = fit_predict_naive_tissue_drug(split_type, fold)
        elapsed = time.time() - start
        save_predictions(split_type, "none", "NaiveTissueDrugMeanPredictor", fold_i, out)
        log_result(split_type, fold_i, "none", "NaiveTissueDrugMeanPredictor", out, elapsed)

LPO fold 0 | none     | NaiveTissueDrugMeanPredictor | r=0.863 rmse=1.361 | params=None | 0s
LPO fold 1 | none     | NaiveTissueDrugMeanPredictor | r=0.866 rmse=1.345 | params=None | 0s
LPO fold 2 | none     | NaiveTissueDrugMeanPredictor | r=0.868 rmse=1.335 | params=None | 0s
LPO fold 3 | none     | NaiveTissueDrugMeanPredictor | r=0.868 rmse=1.339 | params=None | 0s
LPO fold 4 | none     | NaiveTissueDrugMeanPredictor | r=0.865 rmse=1.354 | params=None | 0s
LCO fold 0 | none     | NaiveTissueDrugMeanPredictor | r=0.873 rmse=1.313 | params=None | 0s
LCO fold 1 | none     | NaiveTissueDrugMeanPredictor | r=0.869 rmse=1.335 | params=None | 0s
LCO fold 2 | none     | NaiveTissueDrugMeanPredictor | r=0.858 rmse=1.393 | params=None | 0s
LCO fold 3 | none     | NaiveTissueDrugMeanPredictor | r=0.869 rmse=1.330 | params=None | 0s
LCO fold 4 | none     | NaiveTissueDrugMeanPredictor | r=0.868 rmse=1.345 | params=None | 0s
LDO fold 0 | none     | NaiveTissueDrugMeanPredictor | r=0.202 rmse=3.

In [20]:
out

{'n_train': 141085,
 'n_test': 35112,
 'pearson_r': np.float64(0.8464718593029434),
 'rmse': 1.4565140563124288,
 'test_idx': array([    15,     24,     25, ..., 176188, 176193, 176195],
       shape=(35112,)),
 'y_test': array([-3.183074,  0.387087, -4.008465, ..., 10.271422, 10.243028,
         9.959085], shape=(35112,)),
 'pred_test': array([-5.30267847, -2.87909306, -5.30267847, ..., 10.00242865,
        10.00242865, 10.00242865], shape=(35112,))}

### Run — `RandomForest` (RNA-only, then protein-only)

In [62]:
for split_type, folds in run_splits.items():
    print(f"\n=== {split_type.upper()} ===")
    for fold_i, fold in enumerate(folds):
        print(f"\n--- Fold {fold_i} ---")
        for arm in ARMS:
            print(f"\n--- {arm.upper()} ---")
            params = selected_params["RandomForest"][arm]
            start = time.time()
            out = fit_predict("RandomForest", split_type, fold, arm, params)
            elapsed = time.time() - start
            save_predictions(split_type, arm, "RandomForest", fold_i, out)
            log_result(split_type, fold_i, arm, "RandomForest", out, elapsed, params)
        break


=== LCO ===

--- Fold 0 ---

--- RNA ---
  fitting RandomForest on 141020 train rows... 

KeyboardInterrupt: 

### Run — `XGBoost` (RNA-only, then protein-only)

In [ ]:
for split_type, folds in run_splits.items():
    print(f"\n=== {split_type.upper()} ===")
    for fold_i, fold in enumerate(folds):
        print(f"\n--- Fold {fold_i} ---")
        for arm in ARMS:
            print(f"\n--- {arm.upper()} ---")
            params = selected_params["XGBoost"][arm]
            start = time.time()
            out = fit_predict("XGBoost", split_type, fold, arm, params)
            elapsed = time.time() - start
            save_predictions(split_type, arm, "XGBoost", fold_i, out)
            log_result(split_type, fold_i, arm, "XGBoost", out, elapsed, params)


=== LPO ===

--- Fold 0 ---

--- RNA ---


## Save + quick summary

Dedupes on (split, fold, arm, model), keeping the most recent row — safe to
re-run any single model cell above without re-running the others.

In [ ]:
summary = pd.DataFrame(results_rows).drop_duplicates(
    subset=["split", "fold", "arm", "model"], keep="last"
)
summary.to_csv(RESULTS_DIR / "summary.csv", index=False)
summary.groupby(["split", "arm", "model"])[["pearson_r", "rmse"]].mean()